In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import time

/Users/zafirnasim/Documents/emoji-search/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_parquet("hf://datasets/badrex/LLM-generated-emoji-descriptions/data/train-00000-of-00001.parquet")
df.head()

,character,unicode,short description,tags,LLM description
0,🥇,U+1F947,1ST PLACE MEDAL,"[first place, victory, achievement, success, c...","This emoji represents a first place medal, oft..."
1,🥈,U+1F948,2ND PLACE MEDAL,"[medal, silver, second place, achievement, suc...","This emoji represents a silver medal, often us..."
2,🥉,U+1F949,3RD PLACE MEDAL,"[medal, bronze, third place, achievement, spor...","This emoji represents a bronze medal, symboliz..."
3,🆎,U+1F18E,AB BUTTON (BLOOD TYPE),"[blood type, AB, medical, compatibility, trans...",This emoji represents the AB blood type symbol...
4,🏧,U+1F3E7,ATM SIGN,"[ATM, banking, finance, money, transaction, lo...","This emoji represents an ATM sign, often used ..."


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [9]:
def form_prompts():
    result = []
    for index, row in df.iterrows():
        prompt = f"{row["short description"]}, associated with: {", ".join(row["tags"])}"
        result.append(prompt)
    return result

prompts = form_prompts()
embeddings = model.encode(prompts, convert_to_tensor=True)
type(embeddings)
embeddings.shape

torch.Size([5034, 384])

In [25]:
query = "december"
start_time = time.time()
query_embedding = model.encode(query, convert_to_tensor=True)

from sentence_transformers import util
cosine_scores = util.cos_sim(query_embedding, embeddings)
cosine_scores_list = cosine_scores[0].tolist()
top_results = sorted(range(len(cosine_scores_list)), key=lambda i: cosine_scores_list[i], reverse=True)[:5]

print("Top 5 most similar emojis to '{}'".format(query))

for idx in top_results:
    print("{}: {} (Score: {:.4f})".format(df.iloc[idx]["character"], df.iloc[idx]["short description"], cosine_scores_list[idx]))
end_time = time.time()
print("Search took {:.4f} seconds".format(end_time - start_time))

Top 5 most similar emojis to 'december'
⛄: SNOWMAN WITHOUT SNOW (Score: 0.4025)
🎅🏿: SANTA CLAUS DARK SKIN TONE (Score: 0.3294)
🧑🏿‍🎄: MX CLAUS DARK SKIN TONE (Score: 0.3232)
☃️: SNOWMAN (Score: 0.3139)
☃: SNOWMAN (Score: 0.3139)
Search took 0.0472 seconds
